# Training Unit 1 — Document-Level Detector
**Steps 1-3 of the roadmap: Classical baseline → Transformer baseline → Main DeBERTa detector**

كل الخطوات دي بتستخدم بيانات HC3 اللي جهزناها في Unit 1 (`hc3_train.jsonl`, `hc3_val.jsonl`, `hc3_test.jsonl`).

⚠️ **مهم:** الخطوة الأخيرة (DeBERTa) محتاجة GPU. من فوق: `Runtime` → `Change runtime type` → اختار **T4 GPU** قبل ما تبدأ.

In [ ]:
# Cell 1 — install (pinned transformers to avoid 5.x breaking changes)
!pip install -q "transformers==4.46.0" pandas scikit-learn datasets accelerate torch
# NOTE: after this cell runs for the first time, go to Run → Restart Session, then run all cells again.

In [ ]:
# Cell 2 — Kaggle paths + load HC3 splits
# Attach your HC3 dataset via 'Add Input' first (right panel) — replace 'YOUR-DATASET-SLUG' below
import pandas as pd
import os

DATA_DIR = '/kaggle/input/YOUR-DATASET-SLUG'  # <-- change this to your attached dataset's folder name
MODEL_DIR = '/kaggle/working/models'
os.makedirs(MODEL_DIR, exist_ok=True)

train_df = pd.read_json(f'{DATA_DIR}/hc3_train.jsonl', lines=True)
val_df = pd.read_json(f'{DATA_DIR}/hc3_val.jsonl', lines=True)
test_df = pd.read_json(f'{DATA_DIR}/hc3_test.jsonl', lines=True)

print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))
print('label balance (train):', train_df.label.value_counts().to_dict())

---
## Step 1 — Classical Baseline: TF-IDF + Logistic Regression

In [ ]:
# Cell 3 — Step 1: TF-IDF + Logistic Regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
import joblib

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

# class_weight='balanced' handles the ~2:1 human:ai imbalance we flagged earlier
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train, train_df['label'])

val_preds = clf.predict(X_val)
print('=== Step 1 baseline — VALIDATION performance ===')
print(classification_report(val_df['label'], val_preds, digits=3))

test_preds = clf.predict(X_test)
step1_test_acc = accuracy_score(test_df['label'], test_preds)
step1_test_f1 = f1_score(test_df['label'], test_preds, average='macro')
print(f"=== Step 1 baseline — TEST accuracy: {step1_test_acc:.3f}  macro-F1: {step1_test_f1:.3f} ===")

joblib.dump(vectorizer, f'{MODEL_DIR}/tfidf_vectorizer.joblib')
joblib.dump(clf, f'{MODEL_DIR}/logreg_baseline.joblib')
print('Saved Step 1 model to', MODEL_DIR)

---
## Step 2 — Transformer Baseline: DistilBERT

In [ ]:
# Cell 4 — Step 2: DistilBERT fine-tuning
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from datasets import Dataset
import numpy as np

LABEL2ID = {"human": 0, "ai": 1}
ID2LABEL = {0: "human", 1: "ai"}

def to_hf_dataset(df):
    d = df.copy()
    d['labels'] = d['label'].map(LABEL2ID)
    return Dataset.from_pandas(d[['text', 'labels']])

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

distil_name = "distilbert-base-uncased"
distil_tok = AutoTokenizer.from_pretrained(distil_name)

def tokenize_fn(batch):
    return distil_tok(batch['text'], truncation=True, max_length=256, padding='max_length')

train_ds_tok = train_ds.map(tokenize_fn, batched=True)
val_ds_tok = val_ds.map(tokenize_fn, batched=True)
test_ds_tok = test_ds.map(tokenize_fn, batched=True)

distil_model = AutoModelForSequenceClassification.from_pretrained(
    distil_name, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

args_distil = TrainingArguments(
    output_dir=f'{MODEL_DIR}/distilbert_ckpt',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer_distil = Trainer(
    model=distil_model, args=args_distil,
    train_dataset=train_ds_tok, eval_dataset=val_ds_tok,
    compute_metrics=compute_metrics,
)
trainer_distil.train()

In [ ]:
# Cell 5 — Step 2 test evaluation + save
distil_test_metrics = trainer_distil.evaluate(test_ds_tok)
print('=== Step 2 (DistilBERT) — TEST metrics ===')
print(distil_test_metrics)

trainer_distil.save_model(f'{MODEL_DIR}/distilbert_final')
distil_tok.save_pretrained(f'{MODEL_DIR}/distilbert_final')
print('Saved Step 2 model to', f'{MODEL_DIR}/distilbert_final')

---
## Step 3 — Main Document Detector: DeBERTa-v3-base

In [ ]:
# Cell 6 — Step 3: DeBERTa-v3-base fine-tuning (this is THE main detector)
deberta_name = "microsoft/deberta-v3-base"
deberta_tok = AutoTokenizer.from_pretrained(deberta_name)

def tokenize_fn_deberta(batch):
    return deberta_tok(batch['text'], truncation=True, max_length=256, padding='max_length')

train_ds_tok_d = train_ds.map(tokenize_fn_deberta, batched=True)
val_ds_tok_d = val_ds.map(tokenize_fn_deberta, batched=True)
test_ds_tok_d = test_ds.map(tokenize_fn_deberta, batched=True)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_name, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)

args_deberta = TrainingArguments(
    output_dir=f'{MODEL_DIR}/deberta_ckpt',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=1e-5,
    warmup_steps=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none',
    # fp16 removed on purpose: DeBERTa-v3's disentangled attention conflicts with fp16 grad scaling
)

trainer_deberta = Trainer(
    model=deberta_model, args=args_deberta,
    train_dataset=train_ds_tok_d, eval_dataset=val_ds_tok_d,
    compute_metrics=compute_metrics,
)
trainer_deberta.train()

In [ ]:
# Cell 7 — Step 3 test evaluation + save (THE main detector checkpoint)
deberta_test_metrics = trainer_deberta.evaluate(test_ds_tok_d)
print('=== Step 3 (DeBERTa-v3-base MAIN DETECTOR) — TEST metrics ===')
print(deberta_test_metrics)

trainer_deberta.save_model(f'{MODEL_DIR}/deberta_main_detector')
deberta_tok.save_pretrained(f'{MODEL_DIR}/deberta_main_detector')
print('Saved MAIN DETECTOR to', f'{MODEL_DIR}/deberta_main_detector')

---
## Summary — compare all three

In [ ]:
# Cell 8 — final comparison table
summary = pd.DataFrame([
    {'Step': '1 - TF-IDF+LogReg', 'Test Accuracy': step1_test_acc, 'Test F1 (macro)': step1_test_f1},
    {'Step': '2 - DistilBERT', 'Test Accuracy': distil_test_metrics['eval_accuracy'], 'Test F1 (macro)': distil_test_metrics['eval_f1_macro']},
    {'Step': '3 - DeBERTa-v3-base (MAIN)', 'Test Accuracy': deberta_test_metrics['eval_accuracy'], 'Test F1 (macro)': deberta_test_metrics['eval_f1_macro']},
])
print(summary.to_string(index=False))
summary.to_csv(f'{MODEL_DIR}/step1_3_comparison.csv', index=False)

print('\n⚠️ Sanity check: each step should beat the one before it.')
print('If DeBERTa does NOT clearly beat DistilBERT, something is off — check learning rate, epochs, or data.')

---
## Done — Steps 1-3 (Document-Level Detector)

الموديل الرئيسي محفوظ في `models/deberta_main_detector/` — ده اللي هنستخدمه في:
- Unit التالي: الـ localization head (Step 4) على AITDNA + OpAI-Bench
- لاحقًا: الـ re-detection loop بعد الـ humanization

▶️ **الخطوة الجاية:** Training Unit 2 — Fine-grained Localization (token/span classifier) على AITDNA + OpAI-Bench.

---
## RAID Exploration — قبل ما نكتب أي كود دمج
بنسحب عينة صغيرة بس من RAID (مش الملف كامل 11.8GB) عشان نشوف توزيع `attack` و`model` قبل ما نكتب كود الدمج مع HC3.

In [ ]:
# Cell 9 — RAID exploration (streaming, small sample only)
!pip install -q datasets

from datasets import load_dataset
import pandas as pd

# streaming=True: ما بيحملش الملف كامل (11.8GB)، بيسحب صف صف
raid_stream = load_dataset("liamdugan/raid", split="train", streaming=True)

SAMPLE_SIZE = 5000  # عينة استكشافية بس، مش الداتا الفعلية للتدريب
sample_rows = []
for i, row in enumerate(raid_stream):
    sample_rows.append(row)
    if i + 1 >= SAMPLE_SIZE:
        break

raid_sample_df = pd.DataFrame(sample_rows)
print(f"Pulled {len(raid_sample_df)} rows (streaming sample)\n")

print("=== توزيع attack ===")
print(raid_sample_df['attack'].value_counts())

print("\n=== توزيع model (chatgpt/gpt4/... أو human) ===")
print(raid_sample_df['model'].value_counts())

print("\n=== عدد صفوف attack='paraphrase' في العينة دي ===")
n_paraphrase = (raid_sample_df['attack'] == 'paraphrase').sum()
print(n_paraphrase)

print("\n=== عدد صفوف human (لو موجودة في عمود model) ===")
if 'human' in raid_sample_df['model'].unique():
    print((raid_sample_df['model'] == 'human').sum())
else:
    print("model column ما فيهاش 'human' في العينة دي — لو محتاجها اتأكد من الأسماء الفعلية اللي طلعت فوق في value_counts")

print("\n=== عينة نصوص attack='paraphrase' (أول 3) ===")
for txt in raid_sample_df[raid_sample_df['attack'] == 'paraphrase']['text'].head(3):
    print('-', str(txt)[:200], '...\n')

---
## ⚠️ Kaggle sessions are ephemeral
`/kaggle/working/` gets wiped when the session ends unless you **Save Version** or zip+download the `models/` folder manually:
```python
!zip -r /kaggle/working/models.zip /kaggle/working/models
```
Then download `models.zip` from the Output tab before you close the notebook.